# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

#### Define the Configuration of the engine

In [5]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


####  Run this cell to set up and start your interactive session.


In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, DateType, 
    TimestampType, BooleanType, DecimalType, StructType
)
import re

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 331c16be-11e7-4139-8938-67ca1cd281cd
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 331c16be-11e7-4139-8938-67ca1cd281cd to get into ready status...
Session 331c16be-11e7-4139-8938-67ca1cd281cd has been created.



#### Configure PySpark

In [2]:
spark = SparkSession.builder \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3://prism-nih-bronze/") \
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .getOrCreate()

sc = spark.sparkContext
glueContext = GlueContext(sc)
job = Job(glueContext)

#### Define the file path

In [3]:
file_path = 's3://<bucket_name>/synthetic_data/nmrr_general_information.csv'

#### Load the csv file

In [4]:
spark_df = spark.read.format('csv').option('header', 'true').option("inferSchema", "true").option('sep', ',').option('multiLine', 'true').load(file_path)

#### Preview table

In [5]:
spark_df.show(10)

+-----------+---------------+--------------------+--------------------+------------------+------------------+-----------+--------------+--------------------+-----------------+--------------+--------------------+----------------+--------------------+-----------------------------+-----------------------------+---------------------+----------------------+-----------------------------------------------------------------+---------------------------------+--------------------+-------------------------------------------------------+-----------------------------------------------------+-------------------------------+---------------------------------------------+--------------------------------+------------------+-----------------+-----------------+-----------------+----------------------+---------------------------+-------------------+-----------------------------------+--------------+--------------+----------------------------+------------------------------+----------------------------------

#### preview data schema

In [6]:
spark_df.printSchema()

root
 |-- Research ID: string (nullable = true)
 |-- NMRR ID: string (nullable = true)
 |-- ISR or IIR: string (nullable = true)
 |-- Research Title: string (nullable = true)
 |-- Public Brief Title: string (nullable = true)
 |-- Title Abbreviation: string (nullable = true)
 |-- Protocol ID: string (nullable = true)
 |-- Research Scope: string (nullable = true)
 |-- Research Type : string (nullable = true)
 |-- RMK Priority Area: string (nullable = true)
 |-- Research Level: string (nullable = true)
 |-- Research Description: string (nullable = true)
 |-- Research Keyword: string (nullable = true)
 |-- Primary Disease Area: string (nullable = true)
 |-- Specific Disease Area-Level 1: string (nullable = true)
 |-- Specific Disease Area-Level 2: string (nullable = true)
 |-- Primary Research Area: string (nullable = true)
 |-- Specific Research Area: string (nullable = true)
 |-- Expected Date of First Subject Enrollment / First Data Collection: string (nullable = true)
 |-- Expected Dat

#### Set helper function to clean & standardise header

In [7]:
def clean_string_to_snake_case(text):
    # 1. Convert the string to lowercase
    text = text.lower()
    
    # 2. Remove any characters that are not alphanumeric or spaces (like parentheses)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # 3. Replace spaces with underscores, and strip trailing/leading whitespace
    text = re.sub(r'\s+', '_', text.strip())
    
    return text

#### Set the column header

In [8]:
original_columns = spark_df.columns

#### Convert the header as per helper function

In [9]:
new_columns = [clean_string_to_snake_case(c)for c in original_columns]

#### Set the new column header

In [10]:
spark_df2 = spark_df.toDF(*new_columns)

#### If name is too long  used the following code to shorten it - for example "timeline_of_revision_by_investigator_in_mrg_total_number_of_days_each_time_revision_required_status_is_assigned_date_of_decision_status_assigned_as_revision_submitted_to_mrg_sec_date_of_decision_status_assigned_as_revision_required_for_pre_approval" change it to "timeline_of_revision_by_investigator_in_mrg"

In [49]:
#long_name = [c for c in spark_df2.columns if c.startswith("timeline_of_revision")][0]

In [50]:
# spark_df2 = spark_df2.withColumnRenamed(long_name, "timeline_of_revision_by_investigator_in_mrg")

In [51]:
# matching_cols = [c for c in spark_df2.columns if c.startswith('timeline')]
# print(f"Actual columns found: {matching_cols}")

#### If requires to change data type 

In [ ]:
# spark_df2 = spark_df2.withColumn("date_of_decision", col("date_of_decision").cast("date"))

#### To change multiple column into the same data type 

In [ ]:
# date_columns = ["initial_submission_date", "latest_decision_date"]

# for c in date_columns:
#    spark_df2 = spark_df2.withColumn(c, col(c).cast("date"))

#### View new data schema

In [11]:
spark_df2.printSchema()

root
 |-- research_id: string (nullable = true)
 |-- nmrr_id: string (nullable = true)
 |-- isr_or_iir: string (nullable = true)
 |-- research_title: string (nullable = true)
 |-- public_brief_title: string (nullable = true)
 |-- title_abbreviation: string (nullable = true)
 |-- protocol_id: string (nullable = true)
 |-- research_scope: string (nullable = true)
 |-- research_type: string (nullable = true)
 |-- rmk_priority_area: string (nullable = true)
 |-- research_level: string (nullable = true)
 |-- research_description: string (nullable = true)
 |-- research_keyword: string (nullable = true)
 |-- primary_disease_area: string (nullable = true)
 |-- specific_disease_arealevel_1: string (nullable = true)
 |-- specific_disease_arealevel_2: string (nullable = true)
 |-- primary_research_area: string (nullable = true)
 |-- specific_research_area: string (nullable = true)
 |-- expected_date_of_first_subject_enrollment_first_data_collection: string (nullable = true)
 |-- expected_date_of_

#### Set the directory for the target file 

In [30]:
database_name = 'prism_bronze'
table_name = 'nmrr_general_information_bronze'
full_table_name = f"glue_catalog.{database_name}.{table_name}"

In [31]:
print(full_table_name)

glue_catalog.prism_bronze.nmrr_general_information_bronze


#### set helper function to create DDL for Athena

In [32]:
def generate_create_table_ddl(df: DataFrame, table_name: str, **kwargs) -> str:
    """
    Generates a Athena CREATE TABLE DDL statement from a Spark DataFrame schema.
    """
    # Mapping from Spark data types to Hive/Athena data types
    type_mapping = {
        StringType: "STRING",
        IntegerType: "INT",
        DoubleType: "DOUBLE",
        DateType: "DATE",
        TimestampType: "TIMESTAMP",
        BooleanType: "BOOLEAN"
    }

    # Start building the DDL
    table_type = kwargs.get('table_type', '')
    # Add a space after EXTERNAL if it exists, otherwise leave blank
    table_type_str = f"{table_type} " if table_type else ""
    ddl = f"CREATE {table_type_str}TABLE IF NOT EXISTS {table_name} (\n" 
    
    # if  want to drop and redo the enntire table schema - for existing table  use ddl = f"CREATE OR REPLACE {table_type_str}TABLE {table_name} (\n"
    # beware as REPLACE will delete all dsata that previously available !!

    # Create column definitions from the schema
    column_defs = []
    for field in df.schema.fields:
      #  if partition_keys := kwargs.get("partition_keys"):
      #     if field in partition_keys:
      #          continue
        
        # 1. Handle StructType
        if isinstance(field.dataType, StructType):
             # This is a simplified example; nested structs would require recursive parsing
            sql_type = "STRUCT<...>"
            
        # 2. Handle DecimalType cleanly using isinstance
        elif isinstance(field.dataType, DecimalType):
            sql_type = f"DECIMAL({field.dataType.precision}, {field.dataType.scale})"
            
        # 3. Handle standard types mapped in the dictionary
        else:
            sql_type = type_mapping.get(type(field.dataType), "STRING")

        # Append the formatted column definition
        column_defs.append(f"  {field.name} {sql_type}")

    # Join columns with commas and close the parenthesis
    ddl += ",\n".join(column_defs)
    ddl += "\n)\n"
    
    # Add PARTITIONED BY clause
    if partition_keys := kwargs.get("partition_keys"):
        partition_str = ", ".join([f"{c} STRING" for c in partition_keys]) # Usually partitions need types in DDL, assuming STRING here
        partition_definition = f"PARTITIONED BY ({partition_str})\n"
        ddl += partition_definition
    
    # Add LOCATION clause (Fixed: Added 'LOCATION' keyword)
    if location := kwargs.get("location"):
        ddl += f"LOCATION '{location}'\n"
        
    # Add TBLPROPERTIES clause
    if tbl_properties := kwargs.get('tbl_properties'):
        props_str = ",\n".join([f"  '{k}'='{v}'" for k, v in tbl_properties.items()])
        ddl += f"TBLPROPERTIES (\n{props_str}\n)"
        
    return ddl

#### Set the target table , file type (iceberg) and target loction

In [33]:
create_ddl = generate_create_table_ddl (
    spark_df2, 
    table_name = full_table_name, 
    location = 's3://prism-nih-bronze/', 
   # partition_keys = ['tahun'], -- if want to set for partition
    tbl_properties = {
        'table_type': 'ICEBERG',
        'format': 'PARQUET',
        'parquet_compression': 'snappy'
    }
)

print(create_ddl)

CREATE OR REPLACE TABLE glue_catalog.prism_bronze.nmrr_general_information_bronze (
  research_id STRING,
  nmrr_id STRING,
  isr_or_iir STRING,
  research_title STRING,
  public_brief_title STRING,
  title_abbreviation STRING,
  protocol_id STRING,
  research_scope STRING,
  research_type STRING,
  rmk_priority_area STRING,
  research_level STRING,
  research_description STRING,
  research_keyword STRING,
  primary_disease_area STRING,
  specific_disease_arealevel_1 STRING,
  specific_disease_arealevel_2 STRING,
  primary_research_area STRING,
  specific_research_area STRING,
  expected_date_of_first_subject_enrollment_first_data_collection STRING,
  expected_date_of_study_completion STRING,
  total_study_duration STRING,
  expected_duration_of_study_enrollment_data_collection STRING,
  expected_no_of_subject_to_be_enrolled_data_collected STRING,
  sample_size_outside_of_malaysia STRING,
  expected_sample_sizedata_collected_worldwide STRING,
  final_enrollmentdata_collected STRING,
  

#### start spark.sql

In [34]:
spark.sql(create_ddl)

DataFrame[]


#### define helper function for data ingestion in bronze 

In [35]:
def generate_merge_query(df: DataFrame, table_name: str, **kwargs) -> str:
    """
    Generates a SQL MERGE INTO statement from a Spark DataFrame.

    This function dynamically constructs a MERGE query to upsert data from a
    source (represented by a temporary view of the DataFrame) into a target table.

    :param df: The source Spark DataFrame containing new data.
    :param table_name: The name of the target table to merge into.
    :param kwargs: Keyword arguments for controlling the merge logic.
        - on_keys (list[str]): A list of column names to use for the join
          condition. This is a mandatory argument.
        - source_view (str): The name for the temporary view to be created from
          the DataFrame. Defaults to 'source_view'.
        - target_alias (str): The SQL alias for the target table. Defaults to 'T'.
        - source_alias (str): The SQL alias for the source view. Defaults to 'S'.
    :return: A formatted SQL MERGE INTO statement as a string.
    :raises ValueError: If 'on_keys' is not provided or is empty.
    """
    # --- 1. Validate and Extract Parameters ---
    on_keys = kwargs.get('on_keys')
    if not on_keys:
        raise ValueError("'on_keys' is a mandatory keyword argument and cannot be empty.")

    source_view = kwargs.get('source_view', f'v_{table_name}')
    target_alias = kwargs.get('target_alias', 'T')
    source_alias = kwargs.get('source_alias', 'S')

    all_columns = df.columns
    update_columns = [col for col in all_columns if col not in on_keys]

    # --- 2. Construct Query Clauses ---

    # ON clause for joining source and target
    # Example: T.Encounter_ID = S.Encounter_ID AND T.Patient_ID = S.Patient_ID
    on_clause = " AND ".join([f"{target_alias}.{key} = {source_alias}.{key}" for key in on_keys])

    # WHEN MATCHED clause to update existing records
    # Example: T.Diagnosis_Code = S.Diagnosis_Code, T.Billing_Amount = S.Billing_Amount
    update_clause = ",\n    ".join([f"{target_alias}.{col} = {source_alias}.{col}" for col in update_columns])
    # WHEN NOT MATCHED clause to insert new records
    # Example: (Encounter_ID, Patient_ID) VALUES (S.Encounter_ID, S.Patient_ID)
    insert_columns = ",\n    ".join([f"{col}" for col in all_columns])
    insert_values = ",\n    ".join([f"{source_alias}.{col}" for col in all_columns])

    # --- 3. Assemble the Final MERGE Statement ---
    merge_sql = f"""
    MERGE INTO {table_name} AS {target_alias}
    USING {source_view} AS {source_alias}
    ON {on_clause}
    WHEN MATCHED THEN
      UPDATE SET
        {update_clause}
    WHEN NOT MATCHED THEN
      INSERT (
        {insert_columns}
      ) VALUES (
        {insert_values}
      )
    """
    return merge_sql

#### create temporary view 

In [36]:
view_name = "nmrr_general_information_temp_view"
spark_df2.createOrReplaceTempView(view_name)

#### create merge query

In [37]:
merge_query = generate_merge_query(spark_df2, full_table_name, on_keys = ['nmrr_id'], source_view = view_name) # on_keys should use column with a primary keys !!

In [38]:
print(merge_query)


    MERGE INTO glue_catalog.prism_bronze.nmrr_general_information_bronze AS T
    USING nmrr_general_information_temp_view AS S
    ON T.nmrr_id = S.nmrr_id
    WHEN MATCHED THEN
      UPDATE SET
        T.research_id = S.research_id,
    T.isr_or_iir = S.isr_or_iir,
    T.research_title = S.research_title,
    T.public_brief_title = S.public_brief_title,
    T.title_abbreviation = S.title_abbreviation,
    T.protocol_id = S.protocol_id,
    T.research_scope = S.research_scope,
    T.research_type = S.research_type,
    T.rmk_priority_area = S.rmk_priority_area,
    T.research_level = S.research_level,
    T.research_description = S.research_description,
    T.research_keyword = S.research_keyword,
    T.primary_disease_area = S.primary_disease_area,
    T.specific_disease_arealevel_1 = S.specific_disease_arealevel_1,
    T.specific_disease_arealevel_2 = S.specific_disease_arealevel_2,
    T.primary_research_area = S.primary_research_area,
    T.specific_research_area = S.specific_res

#### run spark sql

In [39]:
spark.sql(merge_query)

DataFrame[]
